# 第17回: Computer Use

Computer Use は、エージェントが API ではなく画面（GUI）を通してコンピューターを操作する仕組み。
画面を見て、クリックやキー入力でアプリを動かす。人が使う画面しかないアプリも操作できる。

今回は、Docker コンテナの中の Linux のデスクトップを操作するエージェントを、2つの方式で実装する。

- **スクリーンショット方式**: OpenAI の computer tool。スクリーンショットを見たモデルが、座標でクリックやキー入力を指示する。エージェントは OpenAI Agents SDK で作る。
- **アクセシビリティツリー方式**: Linux のアクセシビリティの仕組み（AT-SPI）で画面の要素をテキストにして渡し、モデルが要素を指定する。エージェントは LangChain で作る。

---

## 0. 環境準備

この Notebook は、Docker コンテナの中で仮想デスクトップを動かす。**Docker が必要**で、Google Colab では実行できない。

| 実行環境 | 準備 |
| --- | --- |
| Dev Container（推奨） | `ai-agent-seminar-session17` の構成で開く。ホストの Docker を共有する |
| ローカル | Python 3.10 以上の Jupyter と Docker |

Notebook は `session17/` をカレントディレクトリとして実行する（仮想デスクトップのイメージを `desktop/` から作る）。

In [ ]:
# @title 必要なパッケージをインストールする
%pip install -q --break-system-packages \
    "openai>=3.35" "openai-agents>=0.26" "langchain>=1.4" "langchain-openai>=1.7"

In [ ]:
# @title APIキーの設定
import getpass
import os

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')

スクリーンショット方式は OpenAI Agents SDK、アクセシビリティツリー方式は LangChain でエージェントを作る。
computer tool の出力を確かめるところだけ、OpenAI SDK を直接使う。

使うモデルを決める。アクセシビリティツリー方式は、これまでの回と同じ `gpt-5.4-mini` を使う。
スクリーンショット方式（computer tool）は、小さいモデルでは失敗しやすいため `gpt-5.5` を使う（第4節で理由を見る）。

In [ ]:
from openai import AsyncOpenAI

MODEL = "gpt-5.4-mini"  # アクセシビリティツリー方式（LangChain）
CU_MODEL = "gpt-5.5"    # スクリーンショット方式（computer tool、Agents SDK）

client = AsyncOpenAI()

---

## 1. Computer Use とは

### エージェントが環境に触れる手段

これまでの回では、エージェントは Tool を通して外の世界に触れてきた。
Tool の中身は、関数・コマンド（第8回）や MCP Server（第13回）だった。
Computer Use では、Tool の中身が「画面とマウス・キーボード」になる。

| 手段 | エージェントが受け取るもの | エージェントが出すもの | 例 |
| --- | --- | --- | --- |
| API・CLI・MCP | 構造化されたデータ（JSON、コマンドの出力） | Tool の呼び出し・コマンド | 第8回のファイル操作・Web 検索、第13回 |
| アクセシビリティツリー方式 | 画面の要素をテキストにしたもの（アクセシビリティツリー。ブラウザでは DOM も） | 「要素 e12 をクリック」 | Playwright MCP、Browser Use、Windows の UI Automation |
| スクリーンショット方式 | スクリーンショット（画面の画像） | 「座標 (250, 450) をクリック」 | OpenAI computer tool、Anthropic Computer Use |

シェルで実行するか、目的ごとの Tool にするか、MCP で提供するかは Tool の配り方の違いで、エージェントが受け取るものと出すものは変わらない。

下の段ほど、どんなアプリでも操作できる。その代わり遅く、トークンを多く使い、間違えやすい。
**API があるなら API を使う**のが基本で、画面の操作は API がないときの手段。

### 知覚・判断・行動のループ

Computer Use のエージェントも、第4回の ReAct と同じループで動く。
違いは、観測がスクリーンショット（画像）に、行動が入力操作に置き換わったこと。

1. モデルにタスクを渡す。モデルは画面を見るため、まずスクリーンショットを求める。
2. 実行側のコードがスクリーンショットを撮り、モデルに返す。
3. モデルは画面を見て、行動の一覧（クリック・文字入力など）を返す。
4. 実行側のコードが行動を順に実行し、その後の画面を撮って返す。
5. モデルが行動を返さなくなったら、タスクは終わり。

[![](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgVFsi44K_44K544KvIl0gLS0-IE1bIuODouODh-ODqzxicj7vvIhjb21wdXRlciB0b29s77yJIl0KICBNIC0tICJjb21wdXRlcl9jYWxsPGJyPmNsaWNrIC8gdHlwZSAvIC4uLiIgLS0-IEhbIuWun-ihjOOBmeOCi-OCs-ODvOODiTxicj54ZG90b29sIOOBquOBqSJdCiAgSCAtLSAi5pON5L2cIiAtLT4gU1si55S76Z2iPGJyPu-8iOS7ruaDs-ODh-OCueOCr-ODiOODg-ODl--8iSJdCiAgUyAtLSAi44K544Kv44Oq44O844Oz44K344On44OD44OIIiAtLT4gSAogIEggLS0gImNvbXB1dGVyX2NhbGxfb3V0cHV0PGJyPu-8iOeUu-WDj--8iSIgLS0-IE0KICBNIC0tICLooYzli5XjgarjgZciIC0tPiBSWyLmnIDntYLlm57nrZQiXQ)](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgVFsi44K_44K544KvIl0gLS0-IE1bIuODouODh-ODqzxicj7vvIhjb21wdXRlciB0b29s77yJIl0KICBNIC0tICJjb21wdXRlcl9jYWxsPGJyPmNsaWNrIC8gdHlwZSAvIC4uLiIgLS0-IEhbIuWun-ihjOOBmeOCi-OCs-ODvOODiTxicj54ZG90b29sIOOBquOBqSJdCiAgSCAtLSAi5pON5L2cIiAtLT4gU1si55S76Z2iPGJyPu-8iOS7ruaDs-ODh-OCueOCr-ODiOODg-ODl--8iSJdCiAgUyAtLSAi44K544Kv44Oq44O844Oz44K344On44OD44OIIiAtLT4gSAogIEggLS0gImNvbXB1dGVyX2NhbGxfb3V0cHV0PGJyPu-8iOeUu-WDj--8iSIgLS0-IE0KICBNIC0tICLooYzli5XjgarjgZciIC0tPiBSWyLmnIDntYLlm57nrZQiXQ)

**computer tool のループ**

### 主な製品・ライブラリ

Computer Use で新しく必要になるのは、次の2つ。

- **モデル**: 画面を見て、次の行動を決める。
- **観測と実行**: 画面（スクリーンショットや DOM）を取り出し、クリックやキー入力を実際に行う。

Tool を呼び、結果を返す、を繰り返すループは、これまでの回のエージェントと変わらない。

#### モデルの機能

スクリーンショットを見て行動を決める、Computer Use 専用の機能。行動を決めるだけで、実行はしない。観測と実行は、利用する側が書く。

| 名前 | モデルが受け取るもの | モデルが出すもの |
| --- | --- | --- |
| [OpenAI computer tool](https://developers.openai.com/api/docs/guides/tools-computer-use)（Responses API の `computer`） | 画面のスクリーンショット | 座標で指定する操作（`click`・`type`・`keypress`・`scroll`・`drag`・`wait` など）。`actions` 配列にまとめて返す |
| [Anthropic Computer Use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/computer-use-tool) | 画面のスクリーンショット。細部は `zoom` で拡大 | 座標で指定する操作（`left_click`・`type`・`key` など） |
| [Gemini の computer use](https://ai.google.dev/gemini-api/docs/computer-use)（Gemini API の `computer_use`） | 画面のスクリーンショットと、ブラウザなら現在の URL | 画面を 1000×1000 に正規化した座標で指定する操作（`click`・`type`・`scroll`・`navigate` など） |
| オープンウェイトのモデル（[Fara](https://www.microsoft.com/en-us/research/articles/fara1-5-computer-use-agent)、[UI-TARS](https://github.com/bytedance/UI-TARS) など） | 画面のスクリーンショット | 座標で指定する操作をテキストで出力する。形式はモデルごとに違い、読み取る処理も利用する側で書く。モデルの配信（vLLM など）も自分で行う |

#### 観測と実行のライブラリ・ツール

観測するものは、方式によって違う。

- **スクリーンショット方式**: スクリーンショットを観測する。上の表のモデルと組み合わせ、モデルが返した座標でマウスとキーボードを操作する。
- **アクセシビリティツリー方式**: アクセシビリティツリーを観測する。モデルに Computer Use 専用の機能は要らない。ライブラリが定義した Tool（「要素 12 をクリック」など）を、通常の Function Calling で呼ばせる。要素は、ブラウザでは DOM やアクセシビリティツリーから、デスクトップのアプリでは OS のアクセシビリティ API（Windows の UI Automation、Linux の AT-SPI、macOS の Accessibility API）から取り出す。

| 名前 | 操作対象 | 対応環境 | 観測（モデルに渡すもの） | 実行 |
| --- | --- | --- | --- | --- |
| [Playwright](https://playwright.dev/python/) | ブラウザ | Chromium・Firefox・WebKit | スクリーンショット（`page.screenshot()`）、またはアクセシビリティツリー（`aria_snapshot(mode="ai")`。要素に `ref` が付く） | 座標で `page.mouse`・`page.keyboard`、または `ref` で指定した要素を `locator("aria-ref=…")` で操作 |
| [Playwright MCP](https://playwright.dev/mcp/introduction)（MCP Server） | ブラウザ | Chromium・Firefox・WebKit | アクセシビリティツリーのスナップショット | `ref` で指定する Tool 呼び出し |
| [Browser Use](https://github.com/browser-use/browser-use)（エージェントのループも含む） | ブラウザ | Chromium | 操作できる要素に番号を振った DOM のテキスト（スクリーンショットも併用できる） | 要素の番号で指定した操作を、CDP で送る |
| [PyAutoGUI](https://github.com/asweigart/pyautogui) | デスクトップ | Windows・macOS・Linux | スクリーンショット（`screenshot()`） | `click()`・`write()`・`hotkey()` など |
| [xdotool](https://github.com/jordansissel/xdotool) | デスクトップ | Linux（X11） | なし（ImageMagick の `import` などで撮る） | `mousemove`・`click`・`type`・`key` |
| [pywinauto](https://github.com/pywinauto/pywinauto) | デスクトップ | Windows | UI Automation のアクセシビリティツリー（スクリーンショットも撮れる） | 要素を指定した操作、またはマウス・キーボードの入力 |
| [pyatspi](https://gitlab.gnome.org/GNOME/pyatspi2)（AT-SPI） | デスクトップ | Linux | AT-SPI のアクセシビリティツリー | 要素のアクション（押す・文字を入れるなど） |
| [Cua](https://github.com/trycua/cua) | デスクトップ | macOS・Linux・Windows・Android の仮想マシン・コンテナ（環境そのものも用意する） | 環境の中のスクリーンショット | 環境の中のマウス・キーボード |

#### 完成品のサービス

[Operator](https://openai.com/index/introducing-operator/) / [ChatGPT agent](https://openai.com/index/introducing-chatgpt-agent/) は、モデルと観測と実行を、OpenAI 側の仮想ブラウザごとエージェントとして提供する（Operator は 2025年7月に ChatGPT agent に統合された）。
利用者は依頼を書くだけで、中でモデルが何を受け取り何を出しているかは見えない。

#### スクリーンショット方式とアクセシビリティツリー方式

モデルが受け取るものがスクリーンショットかアクセシビリティツリーかで、2つの方式に分かれる。

- **スクリーンショット方式**: Canvas やリモートデスクトップの画面など、アクセシビリティツリーがない画面も操作できる。座標がずれやすく、画像の分トークンを使う。
- **アクセシビリティツリー方式**: 要素を参照で正確に指定でき、安い。アプリがアクセシビリティツリー（ブラウザでは DOM）を出していないと使えない。

### 本回で作るもの

経費の処理を題材にする。仮想デスクトップには、次のファイルとアプリがある。

- ファイルマネージャ（Thunar）: 「受信」フォルダに、ホテルの領収書（テキストファイル）がある。
- テキストエディタ（Mousepad）: 経費一覧（`経費一覧.csv`）を開いている。

エージェントは、領収書を開いて読み、経費一覧に1行追加して保存する。
2つのアプリを行き来し、ファイルの保存という取り消しにくい操作を含む作業になる。

| 節 | 内容 | 観測 | 行動 |
| --- | --- | --- | --- |
| 2 | 仮想デスクトップを用意する | — | — |
| 3 | 画面を操作するクラスを作る | スクリーンショット | xdotool の入力 |
| 4 | スクリーンショット方式（computer tool × OpenAI Agents SDK） | 画像 | 座標 |
| 5 | アクセシビリティツリー方式（AT-SPI × LangChain） | AT-SPI のアクセシビリティツリー（テキスト） | 要素の参照 |
| 6 | 失敗とその対策 | — | — |
| 7 | 安全に動かす（プロンプトインジェクション、取り消しにくい操作の前の確認） | — | — |

---

## 2. 仮想デスクトップを用意する

自分の PC の画面をそのまま操作させるのは危険なので、Docker コンテナの中に仮想のデスクトップを作り、その中だけを操作させる。

| 部品 | 役割 |
| --- | --- |
| Xvfb | 画面のない仮想ディスプレイ（1280×800） |
| Xfce | デスクトップ環境（ウィンドウ、タスクバー） |
| Thunar、Mousepad | 操作するアプリ（ファイルマネージャ、テキストエディタ） |
| ImageMagick の `import` | 画面全体のスクリーンショットを撮るコマンド |
| xdotool | マウスとキーボードの入力を送るコマンド（Windows の SendInput にあたる） |
| AT-SPI | 画面の要素を読む仕組み（Windows の UI Automation にあたる。第5節） |

構成は `desktop/Dockerfile` と `desktop/start.sh`。コンテナは次の設定で起動し、エージェントの操作の影響をコンテナの中にとどめる
（第10回のサンドボックスと同じ考え方）。

- `--network none`: 外部ネットワークに出られない。
- ホストのファイルや Docker ソケットをマウントしない。
- `--cap-drop ALL`、`--security-opt no-new-privileges`: 特権を持たせない。
- `--memory`、`--pids-limit`: 使える資源を制限する。

操作はすべて、ホストから `docker exec` でコンテナの中のコマンドを実行して行う。Notebook 側には画面がなくてよい。

In [ ]:
import asyncio
import json
import subprocess
from pathlib import Path

from IPython.display import Image, display

DESKTOP_IMAGE = "ai-agent-seminar-desktop:17"
DESKTOP_CONTAINER = "ai-agent-seminar-desktop"
HOME = "/home/seminar"  # コンテナの中の利用者のホームフォルダ

if subprocess.run(["docker", "info"], capture_output=True).returncode != 0:
    raise RuntimeError("Docker に接続できない。Docker を起動してから実行する")
if not Path("desktop/Dockerfile").exists():
    raise RuntimeError("session17/ をカレントディレクトリにして実行する")
print("Docker を使える")

イメージを作る。初回は Debian のパッケージを取得するため数分かかる（イメージはおよそ 700 MB）。

In [ ]:
subprocess.run(["docker", "build", "-f", "desktop/Dockerfile", "-t", DESKTOP_IMAGE, "."], check=True)

コンテナの中でコマンドを実行する関数と、コンテナを起動し直す関数を作る。
起動し直すと、ファイルもウィンドウも最初の状態に戻る（`--rm` で起動し、止めると変更は消える）。

In [ ]:
async def docker_exec(*args) -> bytes:
    """仮想デスクトップのコンテナの中でコマンドを実行し、標準出力を返す。"""
    proc = await asyncio.create_subprocess_exec(
        "docker", "exec", "--env", "DISPLAY=:99", DESKTOP_CONTAINER, *map(str, args),
        stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
    stdout, stderr = await proc.communicate()
    if proc.returncode != 0:
        raise RuntimeError(f"{args[0]} が失敗した: {stderr.decode(errors='replace').strip()}")
    return stdout


async def reset_desktop(timeout: float = 60) -> None:
    """コンテナを起動し直し、デスクトップとアプリが表示されるまで待つ。"""
    subprocess.run(["docker", "rm", "-f", DESKTOP_CONTAINER], capture_output=True)
    subprocess.run([
        "docker", "run", "-d", "--rm", "--name", DESKTOP_CONTAINER,
        "--network", "none",
        "--cap-drop", "ALL", "--security-opt", "no-new-privileges",
        "--pids-limit", "512", "--memory", "2g", "--shm-size", "512m",
        DESKTOP_IMAGE,
    ], check=True, capture_output=True)
    for _ in range(int(timeout)):
        try:
            await docker_exec("test", "-f", "/tmp/desktop-ready")  # start.sh が準備完了で作るファイル
            await asyncio.sleep(2)  # ウィンドウの描画を待つ
            return
        except RuntimeError:
            await asyncio.sleep(1)
    raise TimeoutError("デスクトップが起動しなかった")


await reset_desktop()
display(Image(await docker_exec("import", "-window", "root", "png:-")))  # ImageMagick の import で画面全体（root）を PNG にして標準出力へ書き、その画像を表示する

結果を確かめるために、コンテナの中のファイルを読む関数も作る。エージェントではなく、このコードが直接読む。

In [ ]:
async def read_file(path: str) -> str:
    return (await docker_exec("cat", f"{HOME}/{path}")).decode()


async def list_files(path: str) -> list[str]:
    return (await docker_exec("ls", "-1", f"{HOME}/{path}")).decode().split()


print(await read_file("経費一覧.csv"))
print("受信:", await list_files("受信"))
print()
print(await read_file("受信/領収書_ビジネスホテル港南.txt"))

領収書の金額は `¥8,800` だが、経費一覧には数字だけで書く。日付は発行日（11月21日）ではなく、宿泊した日（11月20日）を書く。
エージェントには、こうした値を画面から読み取って判断させる。

---

## 3. 画面を操作するクラスを作る

### 行動と xdotool のコマンド

computer tool が返す行動は次の9種類。座標はスクリーンショットのピクセルで数え、左上が (0, 0)。
行動ごとにメソッドを持つクラスを作り、中身を xdotool のコマンドにする。
第4節の Agents SDK も、第5節の Tool も、このクラスのメソッドを呼んで画面を操作する。

| 行動 | 主な引数 | メソッド | xdotool のコマンド |
| --- | --- | --- | --- |
| `click` | `x`, `y`, `button`（left / right / wheel など）, `keys`（押したままにする修飾キー） | `click` | `mousemove X Y click 1`（右クリックは 3） |
| `double_click` | `x`, `y` | `double_click` | `mousemove X Y click --repeat 2 1` |
| `move` | `x`, `y` | `move` | `mousemove X Y` |
| `drag` | `path`（座標の列） | `drag` | `mousemove` → `mousedown 1` → `mousemove` … → `mouseup 1` |
| `scroll` | `x`, `y`, `scroll_x`, `scroll_y` | `scroll` | ホイールをボタン 4（上）/ 5（下）のクリックとして送る |
| `type` | `text` | `type` | `type -- テキスト`（日本語も入力できる） |
| `keypress` | `keys`（例: `["CTRL", "S"]`） | `keypress` | `key ctrl+s`。キー名は X11 の名前（`Return`、`BackSpace` など）に変換する |
| `wait` | — | `wait` | 少し待つ（ここでは2秒） |
| `screenshot` | — | `screenshot` | `import -window root png:-` |

In [ ]:
XDOTOOL_KEYS = {
    "ENTER": "Return", "RETURN": "Return", "ESC": "Escape", "ESCAPE": "Escape", "TAB": "Tab",
    "SPACE": "space", "BACKSPACE": "BackSpace", "DELETE": "Delete", "DEL": "Delete",
    "HOME": "Home", "END": "End", "PAGEUP": "Page_Up", "PAGEDOWN": "Page_Down",
    "UP": "Up", "DOWN": "Down", "LEFT": "Left", "RIGHT": "Right",
    "ARROWUP": "Up", "ARROWDOWN": "Down", "ARROWLEFT": "Left", "ARROWRIGHT": "Right",
    "CTRL": "ctrl", "CONTROL": "ctrl", "SHIFT": "shift", "ALT": "alt", "OPTION": "alt",
    "META": "super", "CMD": "super", "COMMAND": "super", "SUPER": "super",
}
XDOTOOL_BUTTONS = {"left": 1, "wheel": 2, "middle": 2, "right": 3, "back": 8, "forward": 9}


def xdotool_key(key: str) -> str:
    return XDOTOOL_KEYS.get(key.upper(), key.lower() if len(key) == 1 else key)


class DesktopComputer:
    """仮想デスクトップを docker exec 経由で操作する。ホストの画面や入力デバイスには触れない。"""

    async def xdotool(self, *args):
        await docker_exec("xdotool", *args)

    async def screenshot(self) -> bytes:
        return await docker_exec("import", "-window", "root", "png:-")

    def _modifiers(self, keys, action: str) -> list[str]:
        return [part for key in keys or [] for part in (action, xdotool_key(key))]

    async def click(self, x, y, button="left", keys=None):
        await self.xdotool("mousemove", x, y, *self._modifiers(keys, "keydown"),
                           "click", XDOTOOL_BUTTONS[button], *self._modifiers(keys, "keyup"))

    async def double_click(self, x, y, keys=None):
        await self.xdotool("mousemove", x, y, *self._modifiers(keys, "keydown"),
                           "click", "--repeat", 2, 1, *self._modifiers(keys, "keyup"))

    async def move(self, x, y):
        await self.xdotool("mousemove", x, y)

    async def drag(self, path):
        (x0, y0), *rest = path
        args = ["mousemove", x0, y0, "mousedown", 1]
        for x, y in rest:
            args += ["mousemove", x, y]
        await self.xdotool(*args, "mouseup", 1)

    async def scroll(self, x, y, scroll_x, scroll_y):
        # ホイール1目盛りをおよそ 100px とみなす。上下は 4/5、左右は 6/7 のボタン
        args = ["mousemove", x, y]
        for delta, negative, positive in ((scroll_y, 4, 5), (scroll_x, 6, 7)):
            if delta:
                args += ["click", negative if delta < 0 else positive] * max(1, abs(round(delta / 100)))
        await self.xdotool(*args)

    async def type(self, text):
        await self.xdotool("type", "--delay", 20, "--", text)

    async def keypress(self, keys):
        await self.xdotool("key", "--clearmodifiers", "+".join(xdotool_key(k) for k in keys))

    async def wait(self, seconds=2.0):
        await asyncio.sleep(seconds)


computer = DesktopComputer()

### 座標とスクリーンショット

モデルを使う前に、メソッドを直接呼んで動きを確かめる。領収書のファイルの座標は、上のスクリーンショットから読み取れる。
ダブルクリックすると、テキストエディタの新しいタブで領収書が開く。

In [ ]:
await computer.double_click(290, 171)
await asyncio.sleep(1)
display(Image(await computer.screenshot()))

座標は**モデルに渡したスクリーンショットのピクセル**で数える。次の点に注意する。

- 画面の大きさを固定する。画面やウィンドウの大きさが変わると、同じ要素の座標も変わる。
- 画像を縮小して送ったら、モデルが返す座標を元の大きさに戻してから実行する。本回は 1280×800 をそのまま送るため、変換しない。
- 高解像度ディスプレイ（拡大率 2 倍など）では、スクリーンショットのピクセルと操作の座標がずれることがある。

試したので、デスクトップを最初の状態に戻す。

In [ ]:
await reset_desktop()

---

## 4. スクリーンショット方式（computer tool × OpenAI Agents SDK）

### computer tool の出力を確かめる

ループはライブラリに任せる前に、computer tool が何を返すかを OpenAI SDK で直接確かめる。
Responses API の `tools` に `{"type": "computer"}` を渡す。画面の大きさなどの設定はない。

In [ ]:
response = await client.responses.create(
    model=CU_MODEL,
    tools=[{"type": "computer"}],
    input="テキストエディタで開いているファイルに、何行のデータがあるか教えてください。",
)
call = next(item for item in response.output if item.type == "computer_call")
print(call.model_dump_json(indent=2, exclude_none=True))

モデルは画面を見ていないので、まず `screenshot` を求める。`computer_call` の要点は次のとおり。

- `call_id`: スクリーンショットを返すときに、どの呼び出しへの応答かを示す ID。
- `actions`: 実行してほしい行動の一覧。1回に複数の行動をまとめて返すことがある。
- `pending_safety_checks`: API 側の安全確認（第7節）。空でなければ、実行する前に人の確認が要る。

行動を実行したら、その後の画面を `computer_call_output` で返す。`previous_response_id` で会話を続けるので、
それまでの履歴を自分で送り直す必要はない。スクリーンショットを1回返して、次の応答を見る。

In [ ]:
import base64

png = await computer.screenshot()
response = await client.responses.create(
    model=CU_MODEL,
    tools=[{"type": "computer"}],
    previous_response_id=response.id,
    input=[{
        "type": "computer_call_output",
        "call_id": call.call_id,
        "output": {"type": "computer_screenshot", "image_url": "data:image/png;base64," + base64.b64encode(png).decode()},
    }],
)
for item in response.output:
    if item.type == "computer_call":
        print("computer_call:", [action.model_dump(exclude_none=True) for action in item.actions])
    elif item.type == "message":
        print("回答:", response.output_text)

画面を見て答えられれば、行動を返さずに回答する。続けて操作が必要なら、また `computer_call` を返す。
この「行動を実行してスクリーンショットを返す」を、`computer_call` が返らなくなるまで繰り返すのがエージェントのループ。

### OpenAI Agents SDK の内部処理

ループは OpenAI Agents SDK に任せる。操作する側は、`AsyncComputer` を継承したクラスに、行動と同じ名前のメソッドを書いて渡す。
`Runner.run` の中では、おおまかに次の処理が行われる。

[![](https://mermaid.ink/img/c2VxdWVuY2VEaWFncmFtCiAgcGFydGljaXBhbnQgTiBhcyBOb3RlYm9vawogIHBhcnRpY2lwYW50IFIgYXMgUnVubmVy77yIQWdlbnRzIFNES--8iQogIHBhcnRpY2lwYW50IE0gYXMg44Oi44OH44Or77yIY29tcHV0ZXIgdG9vbO-8iQogIHBhcnRpY2lwYW50IEMgYXMgU2RrRGVza3RvcO-8iEFzeW5jQ29tcHV0ZXLvvIkKICBwYXJ0aWNpcGFudCBEIGFzIOS7ruaDs-ODh-OCueOCr-ODiOODg-ODlwogIE4tPj5SOiBSdW5uZXIucnVuKGFnZW50LCDjgr_jgrnjgq8pCiAgUi0-Pk06IOOCv-OCueOCryArIHRvb2xzOiBjb21wdXRlcgogIGxvb3AgY29tcHV0ZXJfY2FsbCDjgYzov5TjgovplpMKICAgIE0tLT4-UjogY29tcHV0ZXJfY2FsbO-8iGFjdGlvbnM6IGNsaWNrLCB0eXBlLCAuLi7vvIkKICAgIFItPj5DOiBjbGljayh4LCB5KSAvIHR5cGUodGV4dCkgLyAuLi4KICAgIEMtPj5EOiBkb2NrZXIgZXhlYyB4ZG90b29sIC4uLgogICAgUi0-PkM6IHNjcmVlbnNob3QoKQogICAgQy0-PkQ6IGRvY2tlciBleGVjIGltcG9ydCAuLi4KICAgIEMtLT4-UjogUE5H77yIYmFzZTY077yJCiAgICBSLT4-TTogY29tcHV0ZXJfY2FsbF9vdXRwdXTvvIjnlLvlg4_vvIkKICBlbmQKICBNLS0-PlI6IOWbnuetlO-8iOihjOWLleOBquOBl--8iQogIFItLT4-TjogcmVzdWx0LmZpbmFsX291dHB1dA)](https://mermaid.ink/img/c2VxdWVuY2VEaWFncmFtCiAgcGFydGljaXBhbnQgTiBhcyBOb3RlYm9vawogIHBhcnRpY2lwYW50IFIgYXMgUnVubmVy77yIQWdlbnRzIFNES--8iQogIHBhcnRpY2lwYW50IE0gYXMg44Oi44OH44Or77yIY29tcHV0ZXIgdG9vbO-8iQogIHBhcnRpY2lwYW50IEMgYXMgU2RrRGVza3RvcO-8iEFzeW5jQ29tcHV0ZXLvvIkKICBwYXJ0aWNpcGFudCBEIGFzIOS7ruaDs-ODh-OCueOCr-ODiOODg-ODlwogIE4tPj5SOiBSdW5uZXIucnVuKGFnZW50LCDjgr_jgrnjgq8pCiAgUi0-Pk06IOOCv-OCueOCryArIHRvb2xzOiBjb21wdXRlcgogIGxvb3AgY29tcHV0ZXJfY2FsbCDjgYzov5TjgovplpMKICAgIE0tLT4-UjogY29tcHV0ZXJfY2FsbO-8iGFjdGlvbnM6IGNsaWNrLCB0eXBlLCAuLi7vvIkKICAgIFItPj5DOiBjbGljayh4LCB5KSAvIHR5cGUodGV4dCkgLyAuLi4KICAgIEMtPj5EOiBkb2NrZXIgZXhlYyB4ZG90b29sIC4uLgogICAgUi0-PkM6IHNjcmVlbnNob3QoKQogICAgQy0-PkQ6IGRvY2tlciBleGVjIGltcG9ydCAuLi4KICAgIEMtLT4-UjogUE5H77yIYmFzZTY077yJCiAgICBSLT4-TTogY29tcHV0ZXJfY2FsbF9vdXRwdXTvvIjnlLvlg4_vvIkKICBlbmQKICBNLS0-PlI6IOWbnuetlO-8iOihjOWLleOBquOBl--8iQogIFItLT4-TjogcmVzdWx0LmZpbmFsX291dHB1dA)

**Agents SDK の computer tool のループ**

- 行動は `actions` の順にメソッドを呼んで実行し、最後に `screenshot()` を呼んで画像を返す。
- `max_turns` でループの回数に上限を設ける。上限に達すると例外（`MaxTurnsExceeded`）で止まる。
- `pending_safety_checks` が付いた行動は、`ComputerTool(on_safety_check=...)` に渡した関数で、応じるかを決める（第7節）。
- モデル名が `gpt-5.4`・`gpt-5.5` で始まると、新しい `computer` Tool で送る（旧仕様は `computer-use-preview`）。

### AsyncComputer を実装する

第3節の `DesktopComputer` を、`AsyncComputer` の形に合わせる。メソッドの名前は同じなので、引数の形を合わせて呼ぶだけ。
`screenshot()` は base64 の文字列で返す。

Agents SDK は、行動を実行した直後に `screenshot()` を呼ぶ。撮るのが早すぎると、まだ切り替わっていない画面を見たモデルが
同じ操作を繰り返すことがあるため、撮る前に `settle` 秒（0.5 秒）待つ。
途中経過を後で見られるように、実行した行動とスクリーンショットを `steps` に記録しておく。

In [ ]:
import time
from dataclasses import dataclass

from agents import Agent, AsyncComputer, ComputerTool, Runner


@dataclass
class Step:
    index: int
    actions: list[str]  # 実行した行動（表示用の文字列）
    screenshot: bytes   # 実行後の画面


class SdkDesktop(AsyncComputer):
    """DesktopComputer を、Agents SDK の AsyncComputer の形に合わせる。"""

    def __init__(self, desktop: DesktopComputer, settle: float = 0.5):
        self.desktop = desktop
        self.settle = settle
        self.steps: list[Step] = []
        self.pending: list[str] = []  # 次のスクリーンショットまでに実行した行動

    @property
    def environment(self):
        return "linux"

    @property
    def dimensions(self):
        return (1280, 800)

    async def screenshot(self) -> str:
        await asyncio.sleep(self.settle)  # 画面の更新を待ってから撮る
        png = await self.desktop.screenshot()
        step = Step(len(self.steps) + 1, self.pending or ["screenshot()"], png)
        self.steps.append(step)
        self.pending = []
        print(f"[{step.index}] " + ", ".join(step.actions))
        return base64.b64encode(png).decode()  # SDK は base64 の文字列を受け取る

    async def click(self, x, y, button, *, keys=None):
        self.pending.append(f"click({x}, {y}{', ' + button if button != 'left' else ''})")
        await self.desktop.click(x, y, button, keys)

    async def double_click(self, x, y, *, keys=None):
        self.pending.append(f"double_click({x}, {y})")
        await self.desktop.double_click(x, y, keys)

    async def scroll(self, x, y, scroll_x, scroll_y, *, keys=None):
        self.pending.append(f"scroll({x}, {y}, dy={scroll_y})")
        await self.desktop.scroll(x, y, scroll_x, scroll_y)

    async def type(self, text):
        self.pending.append(f"type({text!r})")  # repr で、改行や制御文字も見えるようにする
        await self.desktop.type(text)

    async def wait(self):
        self.pending.append("wait()")
        await self.desktop.wait()

    async def move(self, x, y, *, keys=None):
        self.pending.append(f"move({x}, {y})")
        await self.desktop.move(x, y)

    async def keypress(self, keys):
        self.pending.append("keypress(" + "+".join(keys) + ")")
        await self.desktop.keypress(keys)

    async def drag(self, path, *, keys=None):
        self.pending.append(f"drag({path[0]} → {path[-1]})")
        await self.desktop.drag(path)

途中経過を並べて表示する関数と、実行の統計（リクエスト数・トークン数・時間）をまとめる関数を用意する。

In [ ]:
import html as html_lib

from IPython.display import HTML


@dataclass
class Summary:
    final_text: str
    steps: list[Step]
    requests: int
    input_tokens: int
    output_tokens: int
    elapsed: float


def show_steps(steps: list[Step], columns: int = 3) -> None:
    cells = []
    for step in steps:
        actions = "<br>".join(html_lib.escape(a) for a in step.actions)
        image = base64.b64encode(step.screenshot).decode()
        cells.append(f'<td style="vertical-align:top;width:{100 // columns}%"><b>[{step.index}]</b> '
                     f'<code style="font-size:11px">{actions}</code><br>'
                     f'<img src="data:image/png;base64,{image}" style="width:100%;border:1px solid #ccc"></td>')
    rows = ["<tr>" + "".join(cells[i:i + columns]) + "</tr>" for i in range(0, len(cells), columns)]
    display(HTML("<table>" + "".join(rows) + "</table>"))


def print_summary(s: Summary) -> None:
    print(s.final_text)
    print(f"\nステップ {len(s.steps)} / リクエスト {s.requests} 回 / 入力 {s.input_tokens:,} トークン / "
          f"出力 {s.output_tokens:,} トークン / {s.elapsed:.1f} 秒")

### 実行する

タスクを渡して実行する。タスクには金額などの値を書かない。モデルは画面から値を読み取り、アプリを行き来する必要がある。

In [ ]:
TASK = """経費の処理を手伝ってほしい。
1. ファイルマネージャの「受信」フォルダにある領収書のファイルを開いて、内容を読む。
2. テキストエディタで開いている「経費一覧.csv」の末尾に、その領収書の行を1行追加して保存する。
   列は「日付,区分,金額,支払先,目的」。日付は利用日を YYYY-MM-DD で、金額は数字だけで書く。目的は「大阪出張（顧客訪問）」とする。
3. 終わったら、追加した行を報告する。"""


async def run_cu_agent(task: str, desktop: SdkDesktop, *, max_turns: int = 30):
    """computer tool の Agent を作って実行し、結果と統計を返す。"""
    agent = Agent(name="経費アシスタント", model=CU_MODEL, tools=[ComputerTool(computer=desktop)])
    started, first_step = time.perf_counter(), len(desktop.steps)
    result = await Runner.run(agent, task, max_turns=max_turns)
    usage = result.context_wrapper.usage
    return result, Summary(result.final_output, desktop.steps[first_step:], usage.requests,
                           usage.input_tokens, usage.output_tokens, time.perf_counter() - started)


await reset_desktop()
cu_desktop = SdkDesktop(computer)
cu_result, cu_summary = await run_cu_agent(TASK, cu_desktop)
print_summary(cu_summary)

In [ ]:
show_steps(cu_summary.steps)

モデルは1回の `computer_call` で、クリック・キー入力・文字入力をまとめて返すことがある。
画面を1回ずつ確かめるより、リクエストの回数を減らせる。
その代わり、途中で画面が想定と違っても、まとめた行動は最後まで実行される。

モデルの「追加しました」という報告は、そのまま信じない。保存されたファイルを、プログラムで確かめる。

In [ ]:
ORIGINAL_LINES = ["日付,区分,金額,支払先,目的",
                  "2026-11-05,交通費,640,東京メトロ,打ち合わせ（虎ノ門）",
                  "2026-11-12,会議費,3300,喫茶ポプラ,顧客との打ち合わせ"]
EXPECTED_ROW = "2026-11-20,宿泊費,8800,ビジネスホテル港南,大阪出張（顧客訪問）"


def verify_csv(text: str) -> bool:
    """経費一覧.csv の中身を確かめる。既存の行はそのままで、期待どおりの行が1行だけ増えていれば OK。"""
    lines = text.removesuffix("\n").split("\n")
    added = lines[len(ORIGINAL_LINES):]
    checks = {
        "既存の行が変わっていない": lines[:len(ORIGINAL_LINES)] == ORIGINAL_LINES,
        "追加した行が1行だけ（空行なし）": len(added) == 1,
        "追加した行が期待どおり": added[-1:] == [EXPECTED_ROW],
    }
    for name, ok in checks.items():
        print(("OK  " if ok else "NG  ") + name)
    if all(checks.values()):
        print("    追加された行:", repr(added[0]))
    else:  # どこが違うか分かるように、全体を1行ずつ表示する（repr で、空行や見えない文字も分かる）
        for line in lines:
            print("    " + repr(line))
    return all(checks.values())


verify_csv(await read_file("経費一覧.csv"))

試作では、次のような誤りが出た。

- 末尾に改行がある CSV で、さらに `ENTER` を押してから入力し、空行が入った（`gpt-5.4-mini` でも `gpt-5.5` でも起きた）。画面では気付きにくい。
- 領収書を「処理済み」フォルダへ移すよう頼むと、実際には移していないのに「移動した」と報告した（`gpt-5.4-mini`）。

`gpt-5.4-mini` では誤りが多く、`gpt-5.5` では多くの回で正しくできた。画像から座標を読み取る方式は、モデルの性能の差が出やすい。
画面を見て判断するエージェントは、画面に表れない誤りに気付けない。結果は画面ではなく、データで確かめる。

---

## 5. アクセシビリティツリー方式（AT-SPI × LangChain）

### AT-SPI で画面の要素を読む

OS には、スクリーンリーダーなどの支援技術のために、画面の要素を読む仕組みがある（アクセシビリティ API）。
アプリは、自分の画面の要素を木の形で公開する。これをアクセシビリティツリーと呼ぶ。要素ごとに、役割（ボタン・入力欄・メニューなど）、名前、状態、
画面上の位置、できる操作がある。

| OS | 仕組み |
| --- | --- |
| Windows | UI Automation |
| macOS | Accessibility API |
| Linux | AT-SPI |

ブラウザのアクセシビリティツリーも、同じ仕組みでスクリーンリーダーに画面を渡している。
エージェントも同じアクセシビリティツリーを読めば、画面を画像ではなくテキストで受け取れる。

仮想デスクトップには、AT-SPI でアクセシビリティツリーを読み、JSON で返すコマンド `a11y`（`desktop/a11y.py`）を入れてある。
表示されている要素だけを出し、操作できる要素には `[ref=e12]` のような参照を付ける。

In [ ]:
async def a11y_json(*args) -> dict:
    """コンテナの中の a11y コマンドを実行し、出力の JSON を読む。"""
    for _ in range(3):
        out = await docker_exec("a11y", *args)
        if out.strip():
            return json.loads(out)
        await asyncio.sleep(0.5)  # 出力が空のことがあった（原因は未確認）。少し待って読み直す
    raise RuntimeError(f"a11y {' '.join(map(str, args))} の出力が空だった")


async def a11y_snapshot() -> dict:
    """表示されているアクセシビリティツリー。{"text": ツリーのテキスト, "elements": {ref: 役割・名前・位置・ウィンドウ}} を返す。"""
    return await a11y_json("snapshot")


await reset_desktop()
snapshot = await a11y_snapshot()
print(snapshot["text"])

ファイルマネージャのボタン（「戻る」「ホーム」など）、ファイルの一覧、テキストエディタの本文まで、テキストで読める。
入力欄にフォーカスがあると、`[caret=10/128]` のようにカーソルの位置（何文字目か / 全体の文字数）も出す。文字はこの位置に入る。

参照からは、要素の位置と大きさが分かる。

In [ ]:
receipt = next(e for e in snapshot["elements"].values() if e["name"].startswith("領収書_"))
print(receipt)

### LangChain の create_agent の内部処理

この方式では、モデルに Computer Use 専用の機能は要らない。「アクセシビリティツリーを読む」「参照で要素をクリックする」などの Tool を定義し、
通常の Function Calling で呼ばせる。Tool を呼ぶループは、LangChain の `create_agent` に任せる。
`create_agent` が作るグラフの中では、おおまかに次の処理が行われる。

[![](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgUyhbIuOCv-OCueOCryJdKSAtLT4gTVsibW9kZWwg44OO44O844OJPGJyPu-8iENoYXRPcGVuQUnvvIkiXQogIE0gLS0gInRvb2xfY2FsbHMg44GC44KKIiAtLT4gVFsidG9vbHMg44OO44O844OJPGJyPnJlYWRfc2NyZWVuIC8gY2xpY2sgLyB0eXBlX3RleHQgLi4uIl0KICBUIC0tICJUb29sTWVzc2FnZTxicj7vvIhPSyArIOODhOODquODvO-8iSIgLS0-IE0KICBNIC0tICJ0b29sX2NhbGxzIOOBquOBlyIgLS0-IEUoWyLlm57nrZQiXSkKICBUIDwtLiAiYTExeSBzbmFwc2hvdCAvIHhkb3Rvb2w8YnI-77yIZG9ja2VyIGV4ZWPvvIkiIC4tPiBEWygi5Luu5oOz44OH44K544Kv44OI44OD44OXIild)](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgUyhbIuOCv-OCueOCryJdKSAtLT4gTVsibW9kZWwg44OO44O844OJPGJyPu-8iENoYXRPcGVuQUnvvIkiXQogIE0gLS0gInRvb2xfY2FsbHMg44GC44KKIiAtLT4gVFsidG9vbHMg44OO44O844OJPGJyPnJlYWRfc2NyZWVuIC8gY2xpY2sgLyB0eXBlX3RleHQgLi4uIl0KICBUIC0tICJUb29sTWVzc2FnZTxicj7vvIhPSyArIOODhOODquODvO-8iSIgLS0-IE0KICBNIC0tICJ0b29sX2NhbGxzIOOBquOBlyIgLS0-IEUoWyLlm57nrZQiXSkKICBUIDwtLiAiYTExeSBzbmFwc2hvdCAvIHhkb3Rvb2w8YnI-77yIZG9ja2VyIGV4ZWPvvIkiIC4tPiBEWygi5Luu5oOz44OH44K544Kv44OI44OD44OXIild)

**create_agent のループ**

- model ノードは、それまでのメッセージを渡してモデルを呼ぶ。応答に Tool の呼び出し（`tool_calls`）があれば tools ノードへ進む。
- tools ノードは Tool を実行し、結果を `ToolMessage` として会話に足して、model ノードに戻る。
- Tool の呼び出しがなくなったら終わり。`recursion_limit` でループの回数に上限を設ける。

Tool の結果には、操作の後のアクセシビリティツリーを添えて返す。モデルは、それを見て次の操作を決める。

### Tool を定義する

参照で指定した要素は、その中心の座標を xdotool でクリックする。
Windows なら、UI Automation で要素の位置を調べ、SendInput で押すのにあたる。

最後に読んだアクセシビリティツリーを `elements` に残し、参照から位置を引く。途中経過を見られるように、操作ごとに画面を撮って `a11y_steps` に記録する。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

elements: dict = {}         # 最後に読んだアクセシビリティツリー（ref → 役割・名前・位置・ウィンドウ）
a11y_steps: list[Step] = []


async def screen() -> str:
    """アクセシビリティツリーを読み直して、テキストで返す。"""
    snapshot = await a11y_snapshot()
    elements.clear()
    elements.update(snapshot["elements"])
    return "現在の画面の要素:\n" + snapshot["text"]


async def after(action: str) -> str:
    """操作の後、画面が落ち着くのを待って記録し、アクセシビリティツリーを返す。"""
    await asyncio.sleep(0.5)
    step = Step(len(a11y_steps) + 1, [action], await computer.screenshot())
    a11y_steps.append(step)
    print(f"[{step.index}] {action}")
    return "OK\n" + await screen()


async def press(name: str, ref: str) -> str:
    if ref not in elements:
        return f"失敗: {ref} は現在のアクセシビリティツリーにない。read_screen で読み直すこと。"
    e = elements[ref]
    x, y = e["x"] + e["width"] // 2, e["y"] + e["height"] // 2  # 要素の中心を押す
    if name == "double_click":
        await computer.double_click(x, y)
    else:
        await computer.click(x, y, "right" if name == "right_click" else "left")
    return await after(f'{name}({ref} "{e["name"]}")')


@tool
async def read_screen() -> str:
    """画面に表示されているアクセシビリティツリーを読む。操作できる要素には [ref=e12] のような参照が付く。"""
    return await screen()


@tool
async def click(ref: str) -> str:
    """参照で指定した要素をクリックする。"""
    return await press("click", ref)


@tool
async def double_click(ref: str) -> str:
    """参照で指定した要素をダブルクリックする（ファイルを開くなど）。"""
    return await press("double_click", ref)


@tool
async def right_click(ref: str) -> str:
    """参照で指定した要素を右クリックして、メニューを開く。"""
    return await press("right_click", ref)


@tool
async def type_text(text: str) -> str:
    """キーボードで文字を入力する。文字は、フォーカスのある入力欄のカーソルの位置（アクセシビリティツリーの caret=位置/文字数）に入る。
    入力する前に caret を確かめ、必要なら press_key で移動する（文書の末尾は CTRL+END）。"""
    await computer.type(text)
    return await after(f"type_text({text!r})")


@tool
async def press_key(keys: list[str]) -> str:
    """キーを押す。同時に押すキーを並べる（例: ["CTRL", "S"]、["ENTER"]）。"""
    await computer.keypress(keys)
    return await after("press_key(" + "+".join(keys) + ")")


A11Y_TOOLS = [read_screen, click, double_click, right_click, type_text, press_key]

`create_agent` に、モデルと Tool を渡す。モデルは `ChatOpenAI` で、`gpt-5.4-mini` を使う。

モデルは1回の応答で、複数の Tool をまとめて呼ぶことがある（並列の Tool 呼び出し）。`create_agent` は、まとめて呼ばれた Tool を
同時に実行する。画面の操作は順番が大事で、同時にクリックすると結果が乱れるため、`parallel_tool_calls=False` で1回に1つずつ呼ばせる。

In [ ]:
from langchain_openai import ChatOpenAI

A11Y_PROMPT = "あなたは利用者の代わりに画面を操作するアシスタント。まず read_screen で画面の要素を読んでから操作する。"


async def run_a11y_agent(task: str, *, system_prompt: str = A11Y_PROMPT, recursion_limit: int = 60):
    """アクセシビリティツリーの Agent を作って実行し、結果と統計を返す。"""
    model = ChatOpenAI(model=MODEL, model_kwargs={"parallel_tool_calls": False})  # Tool を1つずつ呼ばせる
    agent = create_agent(model, tools=A11Y_TOOLS, system_prompt=system_prompt)
    a11y_steps.clear()
    started = time.perf_counter()
    result = await agent.ainvoke({"messages": [{"role": "user", "content": task}]},
                                 {"recursion_limit": recursion_limit})
    replies = [m for m in result["messages"] if m.type == "ai"]
    return result, Summary(result["messages"][-1].text, list(a11y_steps), len(replies),
                           sum(m.usage_metadata["input_tokens"] for m in replies),
                           sum(m.usage_metadata["output_tokens"] for m in replies),
                           time.perf_counter() - started)


await reset_desktop()
a11y_result, a11y_summary = await run_a11y_agent(TASK)
print_summary(a11y_summary)
print()
verify_csv(await read_file("経費一覧.csv"))

In [ ]:
show_steps(a11y_summary.steps)

### 2つの方式を比べる

同じタスクの結果を並べる。値は実行ごとに変わる。

In [ ]:
def compare(summaries: dict[str, Summary]) -> None:
    print(f"{'方式':<30}{'ステップ':>8}{'リクエスト':>10}{'入力トークン':>14}{'出力トークン':>12}{'秒':>8}")
    for name, s in summaries.items():
        print(f"{name:<30}{len(s.steps):>8}{s.requests:>10}{s.input_tokens:>14,}{s.output_tokens:>12,}{s.elapsed:>8.1f}")


compare({f"スクリーンショット方式（{CU_MODEL}）": cu_summary, f"アクセシビリティツリー方式（{MODEL}）": a11y_summary})

試作では、アクセシビリティツリー方式は `gpt-5.4-mini` でも5回中4回成功した。スクリーンショット方式は、同じモデルでは誤りが多かった。
要素を名前と参照で指定できるため、小さいモデルでも座標の読み違いが起きない。

入力トークンは、この作業ではアクセシビリティツリー方式のほうが多かった。操作のたびにアクセシビリティツリー全体を送り直すうえ、
小さいモデルは操作を1つずつ確かめるため、ステップ数が増える。トークンの単価は `gpt-5.4-mini` のほうが安い。

| 観点 | スクリーンショット方式（computer tool） | アクセシビリティツリー方式（AT-SPI） |
| --- | --- | --- |
| 操作できる対象 | 画面に映るものすべて（アクセシビリティツリーを出さないアプリ、リモート画面も） | アクセシビリティツリーを出しているアプリ |
| 要素の指定 | 座標。ずれると別の要素を押す | 参照。名前と位置が分かる |
| 見えている範囲 | 画面に映っているものだけ | アクセシビリティツリーにある要素。ほかのウィンドウに隠れた要素も含む（第6節） |
| 見た目の情報 | 色・配置・画像を読める | 色や画像の中身は分からない |
| モデル | computer tool に対応したモデル | Function Calling に対応したモデルなら何でもよい |
| ライブラリ | ループは Agents SDK。実行は `AsyncComputer` を実装して渡す | ループは LangChain。観測と実行は Tool として定義する |
| 実行前の検査 | 座標から押す要素を調べる必要がある（第7節） | 押す要素が引数から分かる |

アクセシビリティツリーを使える場面ではアクセシビリティツリー方式が有利で、アクセシビリティツリーがない画面でスクリーンショット方式を使う。両方を渡すやり方もある。

---

## 6. 失敗とその対策

### よくある失敗

| 失敗 | 原因 | 対策 |
| --- | --- | --- |
| 押したのに画面が変わらない、同じ操作を2回する | 切り替わる前の画面を見た | `wait` を使わせる。行動の後、画面が落ち着いてから撮る（`settle`） |
| 想定外の画面で止まる | ポップアップ、ダイアログ | モデルに閉じさせる。決まったものは実行側で先に閉じる |
| 別の要素を押す | 座標のずれ、ほかのウィンドウに隠れた要素 | 画面の大きさを固定する。押す前に、その座標にある要素を調べる（第7節） |
| 文字が違う場所に入る | カーソルの位置を確かめずに入力した | カーソルの位置をモデルに伝える（AT-SPI の `caret`）。入力の前に `CTRL+END` などで位置を決めるよう、Tool の説明に書く |
| ウィンドウが消える | タスクバーのボタンを押して最小化した | 最小化したウィンドウをアクセシビリティツリーから外し、タスクバーから戻させる |
| 同じ操作を繰り返す | 失敗に気付かない、画面が変わらない | ループの回数に上限を設ける（`max_turns`、`recursion_limit`） |
| 完了したと誤って報告する | 画面の一部だけを見て判断する | 完了の判定をモデルの報告ではなく、プログラムで行う |

Tool の説明しだいで、結果は大きく変わる。試作では、`type_text` の説明に「入力の前にカーソルの位置（caret）を確かめる。
末尾は CTRL+END」と書く前は、`END`（行の末尾）で1行目の末尾に移動してから入力するなどして、4回中2回失敗した。
モデルにとって、Tool の説明は操作の手引きになる。

### ポップアップを試す

作業の途中で、お知らせのダイアログを出す。ダイアログは最前面に出て、画面の操作を邪魔する。
外から起動するアプリは、デスクトップと同じ D-Bus のセッション（アプリ同士がやりとりする仕組み）につなぐ。
つながないと、AT-SPI の接続先が置き換わり、アクセシビリティツリーにダイアログしか現れなくなる。

In [ ]:
async def show_dialog(text: str, title: str = "お知らせ", delay: float = 0) -> None:
    """delay 秒後に、デスクトップにダイアログを出す（desktop/notice.py）。"""
    script = ('export DBUS_SESSION_BUS_ADDRESS="$(cat /tmp/dbus-session-address)"; '  # デスクトップと同じセッション
              f'sleep {delay}; notice "$1" "$2" & '
              'xdotool search --sync --onlyvisible --name "^$1$" windowactivate; wait')  # 最前面に出す
    await docker_exec("bash", "-c", f"({script}) >/dev/null 2>&1 &", "dialog", title, text)


await reset_desktop()
await show_dialog("システムのメンテナンスを 12:00 から行います。作業中のファイルは保存してください。", delay=4)
popup_result, popup_summary = await run_a11y_agent(TASK)
print_summary(popup_summary)
print()
verify_csv(await read_file("経費一覧.csv"))

In [ ]:
show_steps(popup_summary.steps)

モデルはアクセシビリティツリーでダイアログに気付き、「OK」を押してから作業を続ける。
試作では、次のような誤りが何回かに1回出た。どの回も、モデルは「追加した」と報告した。`verify_csv` の結果で確かめる。

- ダイアログを閉じた後、カーソルが CSV の先頭にあるまま入力し、既存の行の前に文字が入った。
- 全角の「）」を、半角の「)」で入力した。

作り込む途中では、次の失敗も起きた。どれも、アクセシビリティツリーが実際の画面と食い違ったことが原因だった。

- ダイアログがほかのウィンドウの後ろに出た。アクセシビリティツリーには「OK」があるので、モデルはその座標を押したが、手前のウィンドウに当たった。
- GTK4 で作ったダイアログ（zenity 4）は、AT-SPI に画面上の位置を正しく返さなかった。「OK」を押したつもりのクリックがタスクバーに当たり、エディタを最小化した。
- 最小化したウィンドウも、AT-SPI では「表示中」のままだった。モデルは見えないエディタに入力しようとした。

アクセシビリティツリーは画面を正確に表すとは限らない。アプリやツールキットの実装しだいで、位置や状態が実際と食い違う。

---

## 7. 安全に動かす

### 画面経由のプロンプトインジェクション

Computer Use のエージェントは、画面に映る文字をすべて読む。Web ページ、メール、文書の中に
エージェント宛ての指示が書かれていると、それに従ってしまうおそれがある（プロンプトインジェクション）。
画面の文字は第三者が書いたもので、利用者の指示ではない。

OpenAI は、computer use の指示（`instructions`）に、画面上の指示を利用者の指示として扱わないことを書くよう勧めている。
指示に書いたうえで、指示だけに頼らず、次の仕組みでも守る。

### 取り消しにくい操作の前で止める

保存、削除、ファイルの移動は取り消しにくい。こうした操作の直前で、人に確認する。
OpenAI のガイドも、送信・削除・購入など影響の大きい操作は、実行の直前に人の確認を求めるよう勧めている。
作業の最初にまとめて確認するのではなく、危険な操作の直前で止める。

座標のクリックは、それだけでは何を押すのか分からない。そこで、実行前に AT-SPI でその座標にある要素を調べる（`a11y at X Y`）。
アクセシビリティツリー方式では、押す要素が引数の参照から分かる。

| 確認する操作 | 調べ方 |
| --- | --- |
| 「保存」「削除」「ゴミ箱」「貼り付け」などの名前の要素を押す | 押す座標（または参照）にある要素の名前 |
| `CTRL+S`（保存） | キーの組み合わせ |
| ファイルマネージャでの `DELETE`、`CTRL+V`（削除、貼り付けによる移動） | キーの組み合わせと、いま操作しているウィンドウの名前 |
| ファイルを別の場所へドラッグする | ドラッグの始点にある要素 |

### API の安全確認

computer tool は、モデル側でも危険を検出すると `computer_call` の `pending_safety_checks` に安全確認を付けて返す。
Agents SDK では、`ComputerTool(on_safety_check=...)` に渡した関数が安全確認ごとに呼ばれ、`True` を返すと応じたことになる。

最後に、コンテナを止める。`--rm` で起動したので、止めると中の変更はすべて消える。

In [ ]:
subprocess.run(["docker", "rm", "-f", DESKTOP_CONTAINER], capture_output=True)